第一步：导入CGSS2021原始数据

In [ ]:
import pandas as pd
import numpy as np

df_2021 = pd.read_stata("CGSS2021.dta"
                        , convert_categoricals=False, index_col='id')
print(df_2021.head())

第二步：筛选所需的原始特征

In [2]:
df_clean2023 = df_2021[['A61_4', 'A3_1', 'A69', 'A1', 'A7a', 'A68_1', 'A68_2', 
                        'A65','A28_1', 'A28_2', 'A28_3','A28_4','A28_5',
                       'A28_6', 'A31a', 'A31b', 'isurban', 'A62',
                       'A36', 'A35', 'A17', 'A15', 'A61_2', 'A61_1', 'A41',
                       'A67_1', 'A67_2', 'A67_3', 'A67_4', 'A67_5', 'A67_6', 'A67_7',
                       'A67_8', 'A67_9']]

第三步：构造被解释变量

In [3]:
#生成被解释变量：是否购买商业养老保险 1=参加 0=没参加
def clean_com_endow(x):
    if x == 1:
        return 1
    elif x == 2:
        return 0
    else:
        return None

df_clean2023['com_endow'] = df_clean2023['A61_4'].apply(clean_com_endow)

In [4]:
#生成变量：年龄
def clean_age(x):
    if x < 0:
        return None
    else:
        return 2026 - x
    
df_clean2023['age'] = df_clean2023['A3_1'].apply(clean_age)

In [5]:
#生成变量：婚姻状况 0=未婚、离婚或丧偶 1=已婚 
def clean_marriage(x):
    if x == 1 or x == 2 or x == 5 or x == 6 or x == 7:
        return 0
    elif x == 3 or x== 4:
        return 1
    else:
        return None

df_clean2023['marriage'] = df_clean2023['A69'].apply(clean_marriage)

In [6]:
#生成变量：家庭规模 int
def clean_member(x):
    if x == -3:
        return None
    else:
        return x + 1

df_clean2023['member'] = df_clean2023['A1'].apply(clean_member)

In [7]:
#生成变量：受教育情况 1=义务教育阶段及以下 2=高中及专科 3= 本科 4=研究生及以上
def clean_edu(x):
    if x == 1 or x == 2 or x == 3 or x == 4:
        return 1
    elif x == 5 or x == 6 or x == 7 or x == 8 or x == 9 or x== 10:
        return 2
    elif x == 11 or x == 12:
        return 3
    elif x == 13:
        return 4
    else:
        return None

df_clean2023['edu'] = df_clean2023['A7a'].apply(clean_edu)

In [8]:
#生成变量：儿子数量 int
def clean_son(x):
    if x < 0 or x > 97:
        return None
    else:
        return x

df_clean2023['son'] = df_clean2023['A68_1'].apply(clean_son)

In [9]:
#生成变量：女儿数量 int
def clean_daughter(x):
    if x < 0 or x > 97:
        return None
    else:
        return x

df_clean2023['daughter'] = df_clean2023['A68_2'].apply(clean_daughter)

In [10]:
#生成变量：房产数量 int
def clean_house(x):
    if x < 0 or x > 97:
        return None
    else:
        return x

df_clean2023['house'] = df_clean2023['A65'].apply(clean_house)

In [11]:
#生成变量：传统媒体使用频率 对四种传统媒体使用情况取平均 1=很少～～5=非常频繁
#首先对四个问题的回答中的缺失值进行处理
def clean_media_old(x):
    if x < 0 or x > 97:
        return None
    else:
        return x

df_clean2023['a281_clean'] = df_clean2023['A28_1'].apply(clean_media_old)
df_clean2023['a282_clean'] = df_clean2023['A28_2'].apply(clean_media_old)
df_clean2023['a283_clean'] = df_clean2023['A28_3'].apply(clean_media_old)
df_clean2023['a284_clean'] = df_clean2023['A28_4'].apply(clean_media_old)  
    
df_clean2023['media_old'] = df_clean2023[['a281_clean', 'a282_clean', 'a283_clean', 'a284_clean']].mean(axis = 1)

In [12]:
#生成变量：新型媒体使用频率 1=很少～～5=非常频繁
def clean_media_new(x):
    if x < 0 or x > 97:
        return None
    else:
        return x
    
df_clean2023['a285_clean'] = df_clean2023['A28_5'].apply(clean_media_new)
df_clean2023['a286_clean'] = df_clean2023['A28_6'].apply(clean_media_new) 
df_clean2023['media_new'] = df_clean2023[['a285_clean', 'a286_clean']].mean(axis = 1)

In [13]:
#生成变量：社会互动频率 1=从来不～～7=几乎每天
def clean_social(x):
    if x < 0 or x > 97:
        return None
    else:
        return 8 - x   #用更大的数字来表示更高社会互动频率

df_clean2023['a31a_clean'] = df_clean2023['A31a'].apply(clean_social)
df_clean2023['a31b_clean'] = df_clean2023['A31b'].apply(clean_social)

df_clean2023['social'] = df_clean2023[['a31a_clean', 'a31b_clean']].mean(axis = 1)

In [14]:
#生成变量：居住在城市还是村镇 1=城市 0=村镇
def clean_urban(x):
    if x == 2:
        return 0
    elif x == 1:
        return 1 
    else:
        return None 

df_clean2023['urban'] = df_clean2023['isurban'].apply(clean_urban)

In [15]:
#生成变量：家庭年总收入 double 9999996=收入高于百万位数
def clean_family_income(x):
    if x >= 9999997 or x < 0:
        return None
    else:
        return x

df_clean2023['family_income'] = df_clean2023['A62'].apply(clean_family_income)

In [16]:
#生成变量：幸福感 1=非常不幸福～～5=非常幸福
def clean_happy(x):
    if x < 0 or x > 97:
        return None
    else :
        return x

df_clean2023['happy'] = df_clean2023['A36'].apply(clean_happy)

In [17]:
#生成变量：社会公平感 1=非常不公平～～5=非常公平
def clean_fair(x):
    if x < 0 or x > 97:
        return None
    else: 
        return x 

df_clean2023['fair'] = df_clean2023['A35'].apply(clean_fair)

In [18]:
#生成变量：是否感到负面情绪 1=总是～～5=从不
def clean_depression(x):
    if x < 0 or x > 97:
        return None
    else:
        return x 

df_clean2023['depression'] = df_clean2023['A17'].apply(clean_depression)

In [19]:
#生成变量：自评健康 1=很不健康～～5=非常健康
def clean_self_health(x):
    if x < 0 or x > 97:
        return None
    else:
        return x 

df_clean2023['self_health'] = df_clean2023['A15'].apply(clean_self_health)

In [20]:
#生成变量：是否有基本养老保险 1=有 0=没有
def clean_basic_endow(x):
    if x == 1 :
        return 1 
    elif x == 2 :
        return 0 
    else:
        return None

df_clean2023['basic_endow'] = df_clean2023['A61_2'].apply(clean_basic_endow)

In [21]:
#生成变量：是否有基本医疗保险 1=有 0=没有
def clean_basic_medicine(x):
    if x == 1 :
        return 1 
    elif x == 2 :
        return 0 
    else: 
        return None

df_clean2023['basic_medicine'] = df_clean2023['A61_1'].apply(clean_basic_medicine)

In [22]:
#生成变量：养老观念 1=政府负责 2=子女负责 3=老人自己负责 4=三方责任均摊
def clean_opinion_gov(x):
    if x == 1 :
        return 1
    elif x == 2 or x == 3 or x== 4:
        return 0
    else:
        return None


def clean_opinion_kid(x):
    if x == 2 :
        return 1
    elif x == 1 or x == 3 or x== 4:
        return 0
    else:
        return None

def clean_opinion_self(x):
    if x == 3 :
        return 1
    elif x == 1 or x == 2 or x== 4:
        return 0
    else:
        return None

def clean_opinion_all(x):
    if x == 4 :
        return 1
    elif x == 1 or x == 2 or x== 3:
        return 0
    else:
        return None

df_clean2023['opinion_gov'] = df_clean2023['A41'].apply(clean_opinion_gov)
df_clean2023['opinion_kid'] = df_clean2023['A41'].apply(clean_opinion_kid)
df_clean2023['opinion_self'] = df_clean2023['A41'].apply(clean_opinion_self)
df_clean2023['opinion_all'] = df_clean2023['A41'].apply(clean_opinion_all) 

In [23]:
#生成变量：是否参与投资活动 1=参加了 0=没参加 
df_clean2023['investment'] = np.where(df_clean2023['A67_1'].eq(1), 0,
                                      np.where(df_clean2023[['A67_2', 'A67_3', 'A67_4', 'A67_5', 
                                   'A67_6', 'A67_7', 'A67_8', 'A67_9']].eq(1).any(axis=1), 
                             1, 
                             np.where(df_clean2023[['A67_2', 'A67_3', 'A67_4', 'A67_5', 
                          'A67_6', 'A67_7', 'A67_8', 'A67_9']].eq(2).any(axis=1), 
                                      0, 
                                      np.nan)))  

第四步：导出清洗好的数据

In [24]:
#导出清洗好的数据
df_final_2021 = df_clean2023[['com_endow', 'age', 'marriage', 'member',
                              'edu', 'son', 'daughter', 'house', 'media_old',
                              'media_new',
                              'social' ,'urban', 'family_income', 'happy',
                              'fair', 'depression' ,'self_health', 'basic_endow',
                              'basic_medicine', 'opinion_gov', 'opinion_kid'
                              ,'opinion_self', 'opinion_all', 'investment']]  
    
df_final_2021.to_excel("2021清洗后数据.xlsx", index = True)   